In [0]:
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown(
    "schema",
    "migration_dev",
    ["migration_dev", "migration_test", "migration_prod_sim"],
)
dbutils.widgets.text("batch_id", "batch_001")

catalog_name = dbutils.widgets.get("catalog").strip()
schema_name = dbutils.widgets.get("schema").strip()
batch_id = dbutils.widgets.get("batch_id").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog_name):
    raise ValueError("Invalid catalog name")

if schema_name not in {
    "migration_dev", "migration_test", "migration_prod_sim"
}:
    raise ValueError("Invalid environment schema")

if not re.fullmatch(r"batch_[0-9]+", batch_id):
    raise ValueError("Invalid batch ID")

table_prefix = f"{catalog_name}.{schema_name}"
volume_root = f"/Volumes/{catalog_name}/{schema_name}"

print(f"Environment: {table_prefix}")
print(f"Batch: {batch_id}")
print(f"Volume root: {volume_root}")


In [0]:
from pyspark.sql import functions as F

pdf_folder = f"{volume_root}/landing/{batch_id}/pdfs"
target_table = f"{table_prefix}.bronze_file_inventory"
pdf_folder = (
    f"/Volumes/workspace/migration_dev/landing/{batch_id}/pdfs"
)
target_table = "workspace.migration_dev.bronze_file_inventory"

files = (
    spark.read
    .format("binaryFile")
    .option("pathGlobFilter", "*.pdf")
    .load(pdf_folder)
)

inventory = files.select(
    F.lit(batch_id).alias("batch_id"),
    F.regexp_extract("path", r"([^/]+)$", 1).alias("file_name"),
    F.regexp_replace("path", r"^dbfs:", "").alias("pdf_path"),
    F.col("length").alias("file_size_bytes"),
    F.col("modificationTime").alias("file_modification_time"),
    F.sha2("content", 256).alias("actual_sha256"),
    F.current_timestamp().alias("inventory_timestamp"),
)

# Refresh only this batch's inventory on each run.
# Other batches in the table remain unchanged.
(
    inventory.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"batch_id = '{batch_id}'")
    .saveAsTable(target_table)
)

saved_inventory = (
    spark.table(target_table)
    .filter(F.col("batch_id") == batch_id)
)

print(f"PDFs inventoried: {saved_inventory.count()}")
display(saved_inventory.orderBy("file_name").limit(5))